In [2]:
import pandas as pd
import sqlite3

# 1. Load raw data from the root folder and push into SQLite
raw_df = pd.read_csv('/online_retail_II.csv')
conn = sqlite3.connect(':memory:')
raw_df.to_sql('raw_orders', conn, index=False, if_exists='replace')

# 2. Clean data using pure SQL
sql_clean_query = """
SELECT DISTINCT
    Invoice,
    StockCode,
    Description,
    Quantity,
    InvoiceDate,
    Price,
    "Customer ID" AS CustomerID,
    Country,
    (Quantity * Price) AS TotalRevenue
FROM raw_orders
WHERE "Customer ID" IS NOT NULL
  AND Quantity > 0
  AND Price > 0
  AND Invoice NOT LIKE 'C%';
"""

clean_df = pd.read_sql_query(sql_clean_query, conn)
print("Cleaned rows ready for cohort analysis:", len(clean_df))
clean_df.head()


Cleaned rows ready for cohort analysis: 779425


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,CustomerID,Country,TotalRevenue
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,30.0


In [3]:
import numpy as np

# 1. Convert InvoiceDate to datetime and extract Year-Month
clean_df['InvoiceDate'] = pd.to_datetime(clean_df['InvoiceDate'])
clean_df['OrderMonth'] = clean_df['InvoiceDate'].dt.to_period('M')

# 2. Find each customer's very first purchase month (Acquisition Cohort)
clean_df['CohortMonth'] = clean_df.groupby('CustomerID')['OrderMonth'].transform('min')

# 3. Calculate Cohort Index (Months since first purchase: 0, 1, 2... 11)
year_diff = clean_df['OrderMonth'].dt.year - clean_df['CohortMonth'].dt.year
month_diff = clean_df['OrderMonth'].dt.month - clean_df['CohortMonth'].dt.month
clean_df['CohortIndex'] = year_diff * 12 + month_diff

# 4. Filter to the first 12 monthly acquisition cohorts & 12-month window
cohorts = sorted(clean_df['CohortMonth'].unique())[:12]
cohort_df = clean_df[(clean_df['CohortMonth'].isin(cohorts)) & (clean_df['CohortIndex'] < 12)].copy()

# 5. Build the Cohort Retention Matrix (%)
cohort_counts = cohort_df.groupby(['CohortMonth', 'CohortIndex'])['CustomerID'].nunique().unstack(1)
cohort_sizes = cohort_counts.iloc[:, 0]
retention_matrix = cohort_counts.divide(cohort_sizes, axis=0).round(3) * 100

print("=== 12-Month Customer Retention Matrix (%) ===")
print(retention_matrix)

# 6. Export clean files to /content/ for easy downloading to Power BI
cohort_df['OrderMonth'] = cohort_df['OrderMonth'].astype(str)
cohort_df['CohortMonth'] = cohort_df['CohortMonth'].astype(str)
cohort_df.to_csv('/content/cleaned_ecommerce_orders.csv', index=False)
retention_matrix.to_csv('/content/cohort_retention_matrix.csv')

print("\nFiles exported successfully!")


=== 12-Month Customer Retention Matrix (%) ===
CohortIndex     0     1     2     3     4     5     6     7     8     9   \
CohortMonth                                                                
2009-12      100.0  35.3  33.4  42.5  38.0  35.9  37.7  34.2  33.6  36.2   
2010-01      100.0  20.6  31.1  30.5  26.4  30.0  25.8  23.0  27.9  31.9   
2010-02      100.0  23.8  22.5  29.1  24.6  20.1  19.3  28.6  25.4  27.5   
2010-03      100.0  19.0  23.0  24.2  23.3  20.3  24.6  30.2  27.5  10.8   
2010-04      100.0  19.4  19.4  16.3  18.4  22.4  27.6  26.2  10.5  10.9   
2010-05      100.0  15.7  16.9  17.3  17.7  25.6  21.3  12.6   5.9   8.3   
2010-06      100.0  17.4  18.9  20.4  23.0  28.5  12.6   8.9   8.1  11.9   
2010-07      100.0  15.6  18.3  29.6  29.0  14.0  11.3  14.5  14.5  11.3   
2010-08      100.0  20.4  29.6  32.1  17.3  11.7   9.9  12.3  13.6  13.0   
2010-09      100.0  22.6  23.5  12.3   9.1  10.3  13.6   9.9  12.8  13.2   
2010-10      100.0  25.7  14.6  12.5   8.